In [3]:
!wget -O /content/development.mp4 https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/1080/Big_Buck_Bunny_1080_10s_1MB.mp4

--2026-09-19 14:29:28--  https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/1080/Big_Buck_Bunny_1080_10s_1MB.mp4
Resolving test-videos.co.uk (test-videos.co.uk)... 172.67.192.184, 104.21.60.65, 2606:4700:3036::6815:3c41, ...
Connecting to test-videos.co.uk (test-videos.co.uk)|172.67.192.184|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1046987 (1022K) [video/mp4]
Saving to: ‘/content/development.mp4’

/content/developmen 100%[===================>]   1022K  --.-KB/s    in 0.1s    

2026-09-19 14:29:28 (7.91 MB/s) - ‘/content/development.mp4’ saved [1046987/1046987]



In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import Video, display

In [ ]:
VIDEO_PATH = Path("/content/development.mp4")

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Vision/vision_unit_03/outputs/day_01"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO_PATH = (OUTPUT_DIR / "opencv_roundtrip.mp4")


print("OpenCV version:", cv2.__version__)
print("Input video:", VIDEO_PATH)
print("Output video:", OUTPUT_VIDEO_PATH)

OpenCV version: 5.0.0
Input video: /content/development.mp4
Output video: /content/drive/MyDrive/Vision/vision_unit_03/outputs/day_01/opencv_roundtrip.mp4


**Read video metadata**

In [8]:
def decode_fourcc(raw_fourcc):
    raw_fourcc = int(raw_fourcc)
    
    characters = [
        chr(
        (raw_fourcc >> (8 * index)) & 0xFF
    ) for index in range(4)
    ]
    
    return "".join(characters).strip("\x00")

In [9]:
def read_video_metadata(video_path):
    capture = cv2.VideoCapture(str(video_path))
    
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    source_fps = float(capture.get(cv2.CAP_PROP_FPS))
    reported_frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    
    raw_fourcc = capture.get(cv2.CAP_PROP_FOURCC)
    
    try:
        backend = capture.getBackendName()
    except Exception:
        backend = "unknown"
        
    capture.release()
    
    if width <= 0 or height <= 0:
        raise RuntimeError("Invalid video resolution")

    if (not np.isfinite(source_fps) or source_fps <= 0):
        raise RuntimeError("Invalid source FPS")
    
    duration_seconds = (
        reported_frame_count / source_fps
        if reported_frame_count > 0 else np.nan
    )
    
    return {
        "width": width,
        "height": height,
        "source_fps": source_fps,
        "reported_frames": reported_frame_count,
        "duration_seconds": duration_seconds,
        "codec": decode_fourcc(raw_fourcc),
        "backend": backend
    }

In [10]:
metadata = read_video_metadata(VIDEO_PATH)
metadata_df = pd.DataFrame([metadata])

display(metadata_df.round(3))

,width,height,source_fps,reported_frames,duration_seconds,codec,backend
0,1920,1080,60.0,600,10.0,h264,FFMPEG
